# 01 — Data Extraction

Parse all Freddie Mac SF LLD quarterly zip files (2018–2025) into parquet files,
build the merged panel, and create the model-specific subsets defined in spec §4.6.

**Run All** executes the full pipeline end-to-end.  
Already-processed quarters are skipped automatically.

Expected final outputs in `data/processed/`:
- `origination_YYYYQN.parquet` × 31 files
- `performance_YYYYQN.parquet` × 31 files
- `origination_all.parquet`
- `performance_all.parquet`
- `merged_panel.parquet`
- `panel_logistic_2021_2025.parquet`
- `panel_cph_2018_2025.parquet`

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

sys.path.insert(0, str(Path('..') / 'src'))
import data_extraction as de

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

PROCESSED = Path('..') / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

## Phase 2a — Parse all origination files

In [ ]:
orig_all_path = de.parse_all_origination(years=range(2018, 2026), out_dir=PROCESSED)
print(f'origination_all.parquet → {orig_all_path}')

## Phase 2b — Parse all performance files

In [ ]:
perf_all_path = de.parse_all_performance(years=range(2018, 2026), out_dir=PROCESSED)
print(f'performance_all.parquet → {perf_all_path}')

## Phase 2c — Merge panel and create model-specific subsets

In [ ]:
panel_path = de.build_merged_panel(out_dir=PROCESSED)
subsets = de.create_model_subsets(out_dir=PROCESSED)
print(f'merged_panel.parquet   → {panel_path}')
print(f'panel_logistic         → {subsets["logistic"]}')
print(f'panel_cph              → {subsets["cph"]}')

## Verification — Row counts, schema, default rates

In [ ]:
panel = pd.read_parquet(panel_path)
print(f'Merged panel shape: {panel.shape}')
print(f'(expect 63 cols = 32 orig + 32 perf - 1 shared loan_id)')

In [ ]:
# Performance rows per origination year — expect ~3-6M per year
panel['orig_year'] = pd.to_datetime(
    panel['first_payment_date'].astype(str), format='%Y%m', errors='coerce'
).dt.year
print('Performance rows per origination year (spec §10 Phase 2 check):')
print(panel.groupby('orig_year').size().rename('perf_rows').to_string())

In [ ]:
# Loan-level default rate on 2021-2025 panel — expect ~0.3% (spec §5.2)
logistic = pd.read_parquet(subsets['logistic'])
terminal_mask = logistic['zero_balance_code'].isin(['03', '06', '09'])
dpd90_mask    = pd.to_numeric(logistic['delinquency_status'], errors='coerce').fillna(0) >= 3
logistic['default'] = (terminal_mask | dpd90_mask).astype(int)
default_rate = logistic.groupby('loan_id')['default'].max().mean()
print(f'Loan-level default rate (2021-2025 panel): {default_rate:.4%}')
print('Expect ~0.3%')

In [ ]:
# Terminal events in 2018-2020 cohorts — Cox PH needs resolved foreclosures
cph = pd.read_parquet(subsets['cph'])
terminal_events = cph[cph['zero_balance_code'] == '09'].copy()
terminal_events['orig_year'] = pd.to_datetime(
    terminal_events['first_payment_date'].astype(str), format='%Y%m', errors='coerce'
).dt.year
print('REO/foreclosure rows (zero_balance_code=09) by origination year:')
print(terminal_events.groupby('orig_year').size().to_string())
print('\nIf 2018-2020 shows zero → zero_balance_code parsing error (spec §10 Phase 2 check)')

In [ ]:
# Field completeness on key modeling columns
key_cols = ['credit_score', 'ltv', 'dti', 'orig_interest_rate',
            'current_upb', 'delinquency_status', 'loan_age']
completeness = (1 - panel[key_cols].isna().mean()).map('{:.1%}'.format)
print('Field completeness (non-null %):')
print(completeness.to_string())

In [ ]:
# Output file inventory
print('Files in data/processed/:')
for parquet_file in sorted(PROCESSED.glob('*.parquet')):
    size_mb = parquet_file.stat().st_size / 1e6
    print(f'  {parquet_file.name:<45} {size_mb:>7.1f} MB')